<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/05_representation_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · What do the learned descriptions of walking look like?

Imagine making a card for each walking clip. Each card contains a list
of numbers describing that clip. Two clips with similar lists have
similar **representations**, also called **feature vectors**. S-JEPA
learned how to produce those lists in notebooks 03 and 04.

Here we ask: **do clips with the same dataset label tend to have similar
representations, and did the extra training make that grouping clearer?**
We make two kinds of map and calculate a grouping score called
**silhouette**. Each map dot represents one clip.

The saved results show mixed labels and silhouette scores close to zero.
Extra training did not improve the measured grouping by condition. We
will work through how to read that finding, why it matters, and why it
is different from notebook 04's classification score.

Follow the tutorial in order: identify the clips, understand their
feature vectors, read the maps, understand silhouette, then connect
the evidence across the three notebooks. All numbers and observations
below describe this saved laptop-profile, Fold 0 run.

In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

## Step 1 · Know what one dot represents

Look at **Fold 0's training row** in the table above. This notebook uses
its **51 clips from 24 source videos**: 15 Normal clips, 23 MS clips,
and 13 PD clips. The legend uses blue for Normal, orange for MS, and
green for PD. These colors are the clips' known dataset labels, not
predictions made by a classifier.

A clip may contain several overlapping motion windows. We summarize
those windows into one vector, so a long clip still contributes one
dot. A source video may contribute several clips, however. Those clips
appear as several dots even though they come from the same recording.
In this fold, **13 of the 23 MS clips come from one source**. A cluster
of orange dots could therefore reflect a shared recording as well as
movement associated with the label. The current plots do not identify
sources, so they cannot distinguish these explanations.

These are the encoder's **training clips**. Notebook 05 excludes the
validation and test partitions from feature extraction, scaling, maps,
and silhouette calculations. We are inspecting patterns in material the
encoder has practiced on. We will need held-out evaluation to find out
whether useful patterns transfer to different sources.

## Step 2 · Compare three descriptions of the same clips

The three panels use the same 51 clips in the same order. What changes
is the list of numbers used to describe each clip:

| Name in the code and plots | Description of one clip | Numbers per clip |
|---|---|---:|
| `ssl` | Features from the original 800-update checkpoint | 96 |
| `continued` | Features after a further 400-update training stage | 96 |
| `visibility` | Mean and standard deviation of each landmark's visibility score | 66 |

For the learned descriptions, the helper passes each complete skeleton
window through the saved **target encoder**, the slowly updated teacher.
It averages features from a fixed set of joint-time positions and then
averages across the clip's windows. This is the same feature-extraction
recipe used in notebook 04. The fixed mask selects **output positions
to average**; it does not hide the input joints at this stage. The 96
features are learned numbers, not 96 named gait measurements.

Visibility provides a simpler comparison. For each of 33 landmarks,
we measure its average pose-detector visibility and how much that value
varies during the clip: `33 × 2 = 66` numbers. For example, a frequently
obscured ankle may have a lower average visibility or greater variation.
This description does not use the landmark's x-y coordinates directly.
Camera framing, occlusion, and pose can all affect it, so it is a check
for alternative cues rather than a pure measure of camera quality.

The next cell loads the trained checkpoints and saves these descriptions
in `training_embeddings.npz`, together with clip names, sources, labels,
and split information. It does not train the encoder again. The saved
output, `training clips plotted: 51`, confirms the number of vectors
prepared for each panel.

In [ ]:
import numpy as np
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.splits import fold_run_dir, load_partition_checkpoint
from sjepa.full_experiment import embed_records, nuisance_features
cfg = get_config(); device = pick_device()
RUN_DIR = fold_run_dir(EXP_DIR, registry, cfg, FOLD)
representations = {}
for stage in ['ssl', 'continued']:
    model = build_model(cfg, device=device, repaired=True)
    load_partition_checkpoint(RUN_DIR / f'{stage}.pt', model, cfg, registry, FOLD, stage, device)
    representations[stage] = embed_records(model, train_recs, cfg, device)
representations['visibility'] = nuisance_features(train_recs)
y = [r.label for r in train_recs]
np.savez(RUN_DIR / 'training_embeddings.npz', **representations, labels=np.array(y),
         clips=np.array([r.clip_name for r in train_recs]),
         sources=np.array([r.source_id for r in train_recs]),
         registry_sha256=registry['registry_sha256'], partition='train')
print('training clips plotted:', len(y))

## Step 3 · Turn a long list of numbers into a two-dimensional map

We cannot directly draw a point with 96 coordinates on a flat page.
**t-SNE** is a method that places those points on a map while trying
to preserve nearby relationships. It is a simplified view of the
features, so some information is inevitably lost.

First, `StandardScaler` puts each feature on a comparable scale using
these training clips. For each feature it subtracts its mean and divides
by its standard deviation when that deviation is nonzero. Each of the
three representations gets its own scaler. Then t-SNE makes a separate
map for each one. Its `perplexity=15` controls the scale of neighborhood
relationships, and `random_state=42` fixes the seed. The condition labels
are used only to color the finished map. This code does not tell t-SNE
to create three condition groups. See the [scikit-learn explanation of
t-SNE](https://scikit-learn.org/stable/modules/manifold.html#t-sne).

When looking at each panel:

1. Find a small group of nearby dots. Are its colors similar or mixed?
2. Check whether that pattern holds throughout the panel. One orange
   patch does not mean all MS clips form a distinct group.
3. Compare the pattern of color mixing across panels, rather than the
   exact dot positions. Each map has its own coordinate system.

The horizontal and vertical axes do not represent walking speed,
disease severity, or time. A dot on the right of one panel is not
necessarily more similar to a dot on the right of another panel.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sjepa.viz import scatter_2d
import matplotlib.pyplot as plt
scaled = {name: StandardScaler().fit_transform(E) for name, E in representations.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, E) in zip(axes, scaled.items()):
    xy = TSNE(n_components=2, perplexity=min(15, len(E)-1), random_state=42, init='pca').fit_transform(E)
    scatter_2d(xy, y, ax, f'training only: {name}')
plt.tight_layout(); plt.savefig(RUN_DIR / 'training_tsne.png', dpi=130); plt.show()

### Read the saved t-SNE figure

In the **original `ssl` panel**, blue, orange, and green dots occur
near one another in several regions. There are local patches with
more of one color, but the panel does not show three clean groups
corresponding to the three conditions.

In the **`continued` panel**, the arrangement changes. A small green
group appears near the upper right, but other green dots remain near
orange and blue dots elsewhere. Extra training has changed the feature
geometry; that particular green patch is insufficient evidence that
all PD clips have become easy to distinguish.

In the **visibility panel**, orange dots are relatively common toward
the upper right, while colors overlap elsewhere. This reminds us that
a map can show structure even when it is based only on visibility
summaries. We cannot conclude from this picture that the learned
encoder relies on visibility, or that visibility predicts labels well
on unseen sources. Those are separate questions to test.

## Step 4 · View the features through a second mapping method

**UMAP** makes another two-dimensional map of the same standardized
vectors. Here it uses `n_neighbors=15` and `min_dist=0.3`, settings that
affect neighborhood relationships and how closely dots can pack. It
also uses seed 42. It is a second view of the same evidence, not an
independent dataset or a second classification experiment.

UMAP and t-SNE can make gaps look stronger or split a continuous group
into apparent islands. The [UMAP documentation explains this limitation](https://umap-learn.readthedocs.io/en/latest/clustering.html).
Look for whether the islands actually match the label colors before
interpreting them as condition groups.

In [ ]:
try:
    import umap
except ImportError:
    print('Install umap-learn to enable the optional UMAP view.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, E) in zip(axes, scaled.items()):
        xy = umap.UMAP(n_neighbors=min(15, len(E)-1), min_dist=0.3, random_state=42).fit_transform(E)
        scatter_2d(xy, y, ax, f'training UMAP: {name}')
    plt.tight_layout(); plt.show()

### Read the saved UMAP figure and its warnings

The **original panel** again mixes colors, including within its small
upper group. The **continued panel** has a striking gap between a small
group at the lower left and a larger group on the right. However,
**both sides contain multiple label colors**. That is the key reading:
a large gap in the map does not by itself separate Normal, MS, and PD.

The **visibility panel** also contains all three colors distributed
through the map. Taken together, these figures do not show a consistent
division into three condition-specific groups. They also cannot prove
that no useful information remains in the full feature vectors.

This cell completed and displayed its figure despite the messages above
it. `IProgress not found` concerns the notebook's progress-bar widget.
The `n_jobs` warning concerns UMAP's use of a fixed seed and single-worker
execution. Neither message reports failure of this completed projection.
They do not explain the mixed colors or the low silhouette values below.

## Step 5 · Measure grouping with a silhouette score

Silhouette asks whether clips are closer, on average, to clips with
their own label than to clips in the closest other label group.
Here the groups are the known Normal, MS, and PD labels; the code
does not first discover groups with a clustering algorithm.

For a particular clip, let **a** be its average distance to the other
clips with its label. Let **b** be the smaller of its average distances
to the two other label groups. Its score is:

$$s = rac{b-a}{max(a,b)}.$$

For illustration, suppose an MS clip has `a=2`, average distance 5 to
Normal, and average distance 6 to PD. Then `b=5` and its score is
`(5−2)/5 = 0.6`: it is closer to its own group. If instead `a=5` and
`b=2`, the score is `−0.6`. These are teaching examples, not our results.

A value near **+1** indicates strong separation for that point; near
**0** means similar within-group and nearest-other-group distances;
a **negative** value means the other group is closer on average.
The notebook reports the mean across clips. The [scikit-learn metric
documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
describes this calculation.

Crucially, the code calculates distances in the **standardized original
feature vectors**: 96 dimensions for each learned representation and
66 for visibility. It uses `scaled`, not the two-dimensional `xy` map.
The printed score therefore measures the features behind the picture,
rather than scoring the appearance of the picture itself.

Each clip counts equally in this mean. Unlike notebook 04's validation
score, it does **not** give every source equal total weight or every
condition one third of the final average. Sources with more clips can
influence both the distances and the average more strongly.

In [ ]:
from sjepa.eval import silhouette
for name, E in scaled.items():
    print(f'{name}: training-only silhouette = {silhouette(E, y):.3f}')

## Step 6 · Interpret our three scores

The saved output reports:

| Representation | Training silhouette | Interpretation for this run |
|---|---:|---|
| Original, `ssl` | **−0.004** | Little average separation by condition under this distance measure |
| Extra training, `continued` | **−0.012** | Slightly lower measured separation after continuation |
| Visibility summaries | **−0.004** | Also little average separation by condition |

These values are all close to zero. They provide little evidence that
each condition forms a compact group well separated from the other
conditions in the standardized feature space. A mean near zero can
also combine positive scores for some clips and negative scores for
others; it does not mean every clip has the same relationships.

The original and visibility scores match at the three decimal places
printed here. They are close, not exactly equal. That similarity does
not establish that the two representations encode the same information
or would achieve the same classification score.

Continuation lowers silhouette by about **0.008**. This goes in the
same unfavorable direction as notebook 04's overall validation macro-F1
change, but the two scores measure different things. No uncertainty
estimate or repeat-run comparison is provided for the silhouette change,
so we should not describe it as a statistically established deterioration.

A negative silhouette is possible and valid. It is not negative
accuracy, a percentage of missed clips, or evidence that a label was
entered incorrectly. There is also no fitted classifier in this cell.
Classification can use particular combinations of features even when
the labels do not form compact groups under overall Euclidean distance.

## Step 7 · Connect the results across notebooks 03, 04, and 05

Each notebook asks a different question:

| Notebook | Question | Evidence so far |
|---|---|---|
| 03: training | Does the model improve at predicting teacher features and retain varied embeddings? | Loss falls and effective rank stays well above one |
| 04: validation | Can a fitted classifier use the features to recognize labels on different sources? | Macro-F1 is 0.294 for the original model and 0.276 after continuation; the original model misses both MS validation clips |
| 05: inspection | Do the training features visibly and numerically group clips by condition? | Maps mix colors and training silhouette stays near zero for all three representations |

**The learning process is working, but useful separation of the condition
labels remains weak in the checks performed so far.** Notebook 03 showed
that the model could learn its masked feature-prediction task. That task
never required its features to arrange the three condition labels into
distinct groups. Notebook 05 helps make that distinction visible.

There is no contradiction between seeing some orange neighbors here and
obtaining MS F1 of zero in notebook 04's original model. This figure
contains 23 **training** MS clips, many sharing a source. The two missed
MS clips came from **different validation sources** and do not appear
in these maps. A familiar recording can look consistent within training
while a new recording is still misclassified.

The continued model's extra training has not produced a better overall
result in this example: its validation macro-F1 and training silhouette
are both lower. That supports retaining notebook 04's original checkpoint
under the existing selection rule. It does not identify why transfer is
weak or establish that every longer training recipe would fail.

Visibility is a useful comparison because it asks whether simple
detection-related information also has structure. Its near-zero score
does not prove recording conditions are harmless, and similarity between
maps does not prove the encoder learned a camera shortcut. We still need
controlled comparisons on held-out sources to assess those possibilities.

## Step 8 · Decide what this inspection justifies next

A useful next development check is to label or color training dots by
**source video**, or to inspect a source-balanced sample. That would
help distinguish repeated-recording groups from groups spanning several
independent sources. It is a proposed check, not a result shown here.
Inspecting skeleton quality and the most confusing training examples can
also help formulate a specific improvement to test.

Keep the saved seed and settings as the reference. Trying many projection
settings until the colors look separated would not demonstrate improved
classification. Any sensitivity study should report the variation,
including views that make the grouping look less convincing.

Notebook 06 tests the fixed training-and-selection procedure across five
source-grouped folds, with a fresh model in each fold. Its Random Forest,
visibility, mean-pose, and majority-label comparisons help judge whether
S-JEPA offers an advantage over simpler alternatives. Choose any changes
to the development recipe before using test results to assess it.

The conclusion supported here is: “For 51 training clips from 24 sources,
the original, continued, and visibility representations had silhouette
scores near zero, and both mapping methods showed mixed condition labels.
The additional training stage did not improve this grouping measure.
These are descriptive training results; performance on held-out sources
must be assessed separately.”